# Project 1

Authors:

In [16]:
import os
from openai import OpenAI
from pydantic import BaseModel
from scrapper import load_from_url, save_to_file
import gradio as gr
from pathlib import Path
#from typing import List

# Initialize client
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file in current directory
api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

Key loaded: True


In [ ]:
INPUT_DATA="input_data/mindfulambition-net-beginners-mind.txt"
with open(INPUT_DATA, "rb") as input_file:
    content = input_file.read()
    print(f"Extracted text: {input_file}")

Extracted text: <_io.BufferedReader name='input_data/mindfulambition-net-beginners-mind.txt'>


In [13]:
def get_summary(text_chunk):
    """Sends a single chunk to the LLM for summarization."""
    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": "You are the witty, warm solo host of a podcast called 'The Joy of Learning.' Your job is to turn source material into an engaging spoken-word script — not a summary, a script meant to be read aloud by a text-to-speech voice.\n\nRules:\n1. Open with exactly this greeting style: 'Welcome to The Joy of Learning, the podcast where we explore [general subject in your own words].' Then transition naturally into the episode.\n2. Explain the core concept in your own words — do not summarize the source paragraph by paragraph, and do not reference the article, its author, or where the content came from. Write as if this is your own idea you're excited to share.\n3. Be genuinely funny: use a playful analogy, a light joke, or a self-aware aside at least once. Think 'smart friend explaining something cool over coffee,' not 'lecture.'\n4. Write only what should be spoken aloud — no headers, no stage directions, no markdown, no sound-effect cues.\n5. Keep it under 300 words so the TTS output stays a reasonable length.\n6. End with a short, warm sign-off that invites the listener to try adopting the mindset themselves."},
            {"role": "user", "content": f"Here is the source material to base the episode on:\n\n{text_chunk}"}
        ],
        temperature=0.8
    )
    print(response)
    return response.choices[0].message.content

res = get_summary(INPUT_DATA)  
print(res)


ChatCompletion(id='chatcmpl-E1oX6F7diQ8fkxoIedHVRPzU0XzIg', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Welcome to The Joy of Learning, the podcast where we explore the beauty of embracing new ideas and experiences. Today, we're diving into a concept that’s as refreshing as a cold glass of lemonade on a hot summer day: the Beginner's Mind.\n\nNow, imagine your brain as a sponge. When it’s new, it soaks up information like it’s the last sponge on Earth. But as we grow older and accumulate experiences—like collecting dust bunnies under the couch—we tend to get a bit rigid. We start thinking we know it all. Spoiler alert: we don’t! \n\nThe idea of the Beginner's Mind is all about approaching situations with openness, eagerness, and a lack of preconceptions. It’s like being a curious little kid again. Remember that? When you could ask a million questions about why the sky is blue or how that one weird-looking bug can climb walls like

In [14]:
def text_to_speech_ai(text, output_file="audio_output.mp3"):
    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )
    
    # Save the binary audio stream to a file
    with open(output_file, "wb") as f:
        f.write(response.content)
    print(f"High-quality audio saved to {output_file}")

text_to_speech_ai(res)

High-quality audio saved to audio_output.mp3


In [19]:
DEFAULT_SOURCES = """https://www.infoprolearning.com/blog/understanding-the-learning-curve-why-its-important-in-employee-training-and-development/
https://iopn.library.illinois.edu/pressbooks/instructioninlibraries/chapter/learning-theories-understanding-how-people-learn/
https://mindfulambition.net/beginners-mind/
https://www.cmu.edu/teaching/designteach/design/instructionalstrategies/groupprojects/benefits.html
https://www.youtube.com/watch?v=AWukNG1UpEs"""

VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]

def generate_podcast(
    sources_text: str,
    episode_title: str
):
    sources = [s.strip() for s in sources_text.splitlines() if s.strip()]
    if not sources:
        print("Please enter at least one source (URL or file path).")

    log_lines = []

    def log(msg):
        log_lines.append(msg)
        return "\n".join(log_lines)

    # 1. Load every source
    docs = []
    for i, src in enumerate(sources):
        try:
            doc = load_from_url(src)
            save_to_file(doc)
            docs.append(doc)
            log(f"✓ Loaded ({doc.source_type}): {src}  [{len(doc.raw_text.split())} words]")
        except Exception as e:
            log(f"✗ Failed to load {src}: {e}")
        #progress((i + 1) / len(sources) * 0.3, desc=f"Loaded {i+1}/{len(sources)} sources")

    if not docs:
        print("None of the sources could be loaded. Check the log for details.")

    #progress(0.3, desc="Condensing sources...")
    try:
        script = get_summary(docs)
    except Exception as e:
        print(f"Script generation failed: {e}")

    # 3. Text-to-speech
    #progress(0.75, desc="Generating audio (this can take a minute)...")
    safe_title = "".join(c if c.isalnum() or c in " -_" else "" for c in (episode_title or "episode"))
    output_path = f"output/{safe_title or 'episode'}.mp3"
    try:
        audio_path = text_to_speech_ai(script, output_file=output_path)
    except Exception as e:
        print(f"Audio generation failed: {e}")

    #progress(1.0, desc="Done!")
    return script, audio_path, "\n".join(log_lines)


with gr.Blocks(title="Podcast Studio") as demo:
    gr.Markdown(
        """
        # 🎙️ Podcast Studio
        Turn a handful of sources — web articles, YouTube videos, PDFs, or text notes —
        into a single, cohesive podcast episode script and audio file.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            sources_input = gr.Textbox(
                label="Sources (one per line — URLs, YouTube links, or local .pdf/.txt paths)",
                value=DEFAULT_SOURCES,
                lines=8,
            )
            episode_title_input = gr.Textbox(
                label="Episode title",
                value="The Joy of Learning: Embracing the Curve",
            )
            topic_input = gr.Textbox(
                label="Episode topic",
                value="the learning curve, the cognitive science behind it, beginner's mind, "
                      "and why learning in groups helps",
            )
            audience_input = gr.Textbox(
                label="Target audience",
                value="adult learners currently going through a steep learning curve",
            )
            with gr.Row():
                voice_input = gr.Dropdown(VOICES, value="alloy", label="Voice")
                target_words_input = gr.Slider(
                    400, 2500, value=1300, step=100, label="Target script length (words)"
                )
            generate_btn = gr.Button("Generate Podcast", variant="primary")

        with gr.Column(scale=1):
            audio_output = gr.Audio(label="Episode Audio", type="filepath")
            script_output = gr.Textbox(label="Episode Script", lines=15)
            log_output = gr.Textbox(label="Pipeline Log", lines=6)

    generate_btn.click(
        fn=generate_podcast,
        inputs=[
            sources_input,
            episode_title_input
        ],
        outputs=[script_output, audio_output, log_output],
    )

if __name__ == "__main__":
    Path("output").mkdir(exist_ok=True)
    demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


None of the sources could be loaded. Check the log for details.
ChatCompletion(id='chatcmpl-E1ojMXujTzHDk5X2gMOcV70KwVAQp', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Welcome to The Joy of Learning, the podcast where we explore the wonders of knowledge and the thrill of discovery! Today, let’s dive into the fascinating world of how we learn and adapt throughout our lives.\n\nSo, let’s imagine for a moment that our brains are like sponges—except, of course, these sponges are equipped with Wi-Fi and can download updates at lightning speed! Learning isn’t just about memorizing facts; it’s about making connections, adapting our thoughts, and growing like a fine cheese—getting sharper or sometimes a bit moldy, depending on your choices. \n\nEvery time we encounter new information, our brains shift and rearrange, like a game of Tetris in our heads. The more we learn, the more we build this intricate structure of knowledge! And here’s 